# 04 — Physics-Informed Neural Network (PINN): Inverse Design of Underwater Acoustic Coatings

## Overview
This notebook implements a **Physics-Informed Neural Network** where the TMM acoustic equations are embedded directly into the training loss.

**Key idea**: Beyond parameter MSE and surrogate reconstruction, a **differentiable PyTorch implementation of the Transfer-Matrix Method** is evaluated at a subset of frequencies (10 log-spaced points) on the predicted parameters. The physics loss penalises predicted designs whose TMM-computed absorption deviates from the target.

**Composite loss**: `w1 * MSE_params + w2 * MSE_surrogate + w3 * physics_loss`, with w3 gradually increasing during training (curriculum-style).

This ensures predictions are not just data-consistent but also **physics-consistent**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# ---- Force NVIDIA GPU ----
assert torch.cuda.is_available(), (
    "CUDA not available! Install PyTorch with CUDA support:\n"
    "  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121"
)
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## TMM Forward Model — NumPy version (for final validation)

In [ ]:
RHO_WATER=1000.0; C_WATER=1500.0; Z_WATER=RHO_WATER*C_WATER; W_FILM=2.0
HOLLOW_MAP={1:0,2:1,4:2,5:3,7:4,8:5}
PARAM_COLUMNS=[f'd{i}' for i in range(1,11)]+['m2','m3','m5','m6','m8','m9']+['rho','eta','E','nu']
PARAM_BOUNDS=np.array([[1,20]]*10+[[20,1980]]*6+[[1000,1500],[.1,.8],[1e7,1e8],[.4,.49]])

def tmm_forward(params):
    d_mm=np.asarray(params[:10],dtype=float); m_mm=np.asarray(params[10:16],dtype=float)
    rho_r,eta,E_r,nu=float(params[16]),float(params[17]),float(params[18]),float(params[19])
    d_m,m_m=d_mm/1e3,m_mm/1e3; Ec=E_r*(1+1j*eta)
    lam=(Ec*nu)/((1+nu)*(1-2*nu)); mu=Ec/(2*(1+nu))
    s=0.0
    for f in range(1,1001):
        w=2*np.pi*f; T=np.eye(2,dtype=complex)
        for i in range(10):
            eps=m_m[HOLLOW_MAP[i]]/W_FILM if i in HOLLOW_MAP else 0.0
            re=rho_r*(1-eps**2)
            num=mu*(lam+2*mu)*(eps**2+1)+2*eps**2*lam; den=(lam+mu)*eps**2+mu
            Se=num/den if abs(den)>1e-15 else num*1e15
            ce=np.sqrt(Se/re); k=w/ce; Ze=re*ce
            ck,sk=np.cos(k*d_m[i]),np.sin(k*d_m[i])
            t21=1j*sk/Ze if abs(Ze)>1e-15 else 1e15+0j
            T=T@np.array([[ck,1j*Ze*sk],[t21,ck]],dtype=complex)
        if abs(T[1,0])<1e-15: R=1.0
        else: Zi=T[0,0]/T[1,0]; R=(Zi-Z_WATER)/(Zi+Z_WATER) if abs(Zi)<1e15 else 1.0
        a=1-abs(R)**2; s+=max(0,min(1,a.real if hasattr(a,'real') else a))
    return s/1000

print(f'Base-case: {tmm_forward([10]*10+[1000]*6+[1130,.4,5e7,.44]):.4f}')

## Differentiable TMM in PyTorch (for physics loss)
Operates on batches of parameter vectors at a subset of frequencies. Uses `torch.cfloat`.

In [ ]:
# 10 log-spaced sample frequencies for the physics loss
PHYSICS_FREQS = torch.tensor([10, 30, 60, 100, 200, 350, 500, 700, 850, 1000], dtype=torch.float32)

# Hollow layer lookup: for layer i (0-idx), which index in m_vals (0-5) to use, or -1 if solid
HOLLOW_IDX = torch.tensor([-1, 0, 1, -1, 2, 3, -1, 4, 5, -1], dtype=torch.long)  # 10 layers


def tmm_torch(params_phys, freqs=None):
    """
    Differentiable TMM forward model.

    params_phys: (B, 20) tensor in physical units
        [d1..d10 (mm), m2..m9 (mm), rho, eta, E, nu]
    freqs: 1-D tensor of frequencies (Hz). Default: PHYSICS_FREQS

    Returns: (B,) mean absorption over the given frequencies
    """
    if freqs is None:
        freqs = PHYSICS_FREQS.to(params_phys.device)

    B = params_phys.shape[0]
    F = freqs.shape[0]

    d_m = params_phys[:, :10] / 1000.0          # (B, 10)
    m_m = params_phys[:, 10:16] / 1000.0         # (B, 6)
    rho_r = params_phys[:, 16]                    # (B,)
    eta   = params_phys[:, 17]
    E_r   = params_phys[:, 18]
    nu    = params_phys[:, 19]

    W = 2.0
    Z_w = 1.5e6

    # Complex Young's modulus
    E_re = E_r
    E_im = E_r * eta
    E_c = torch.complex(E_re, E_im)  # (B,)

    # Lame constants
    lam = (E_c * nu) / ((1 + nu) * (1 - 2 * nu))  # (B,)
    mu  = E_c / (2.0 * (1 + nu))                    # (B,)

    omega = 2.0 * np.pi * freqs  # (F,)

    # Expand for broadcasting: (B, F)
    alpha_all = torch.zeros(B, F, device=params_phys.device)

    for fi in range(F):
        w = omega[fi]
        # Transfer matrix: start with identity (B, 2, 2) complex
        T = torch.zeros(B, 2, 2, dtype=torch.cfloat, device=params_phys.device)
        T[:, 0, 0] = 1.0; T[:, 1, 1] = 1.0

        for li in range(10):
            d_i = d_m[:, li]  # (B,)
            hidx = HOLLOW_IDX[li].item()
            if hidx >= 0:
                eps = m_m[:, hidx] / W  # (B,)
            else:
                eps = torch.zeros(B, device=params_phys.device)

            rho_eff = rho_r * (1.0 - eps ** 2)
            eps2 = eps ** 2

            num = mu * (lam + 2*mu) * (eps2 + 1) + 2 * eps2 * lam
            den = (lam + mu) * eps2 + mu
            S_eff = num / den  # (B,) complex

            rho_eff_c = torch.complex(rho_eff, torch.zeros_like(rho_eff))
            c_eff = torch.sqrt(S_eff / rho_eff_c)
            k = w / c_eff
            Z_eff = rho_eff_c * c_eff

            d_i_c = torch.complex(d_i, torch.zeros_like(d_i))
            kd = k * d_i_c
            cos_kd = torch.cos(kd)
            sin_kd = torch.sin(kd)

            # Build per-layer transfer matrix (B, 2, 2)
            Ti = torch.zeros(B, 2, 2, dtype=torch.cfloat, device=params_phys.device)
            Ti[:, 0, 0] = cos_kd
            Ti[:, 0, 1] = 1j * Z_eff * sin_kd
            Ti[:, 1, 0] = 1j * sin_kd / Z_eff
            Ti[:, 1, 1] = cos_kd

            # Batch matrix multiply
            T = torch.bmm(T, Ti)

        T11 = T[:, 0, 0]
        T21 = T[:, 1, 0]
        Z_in = T11 / (T21 + 1e-15)
        R = (Z_in - Z_w) / (Z_in + Z_w)
        alpha = 1.0 - torch.abs(R) ** 2
        alpha = torch.clamp(alpha, 0.0, 1.0)
        alpha_all[:, fi] = alpha

    return alpha_all.mean(dim=1)  # (B,)


# Quick test
_t = torch.tensor([[10]*10+[1000]*6+[1130,.4,5e7,.44]], dtype=torch.float32)
print(f'Torch TMM (10 freqs): {tmm_torch(_t).item():.4f}')

## Data Loading

In [ ]:
df = pd.read_csv('../data/lhs_data.csv')
param_cols=[c for c in df.columns if c!='Average_Absorption']
X=df[param_cols].values; y=df['Average_Absorption'].values

X_tr,X_tmp,y_tr,y_tmp=train_test_split(X,y,test_size=0.2,random_state=42)
X_val,X_te,y_val,y_te=train_test_split(X_tmp,y_tmp,test_size=0.5,random_state=42)

ps=MinMaxScaler(); X_tr_n=ps.fit_transform(X_tr); X_val_n=ps.transform(X_val); X_te_n=ps.transform(X_te)
as_=MinMaxScaler(); y_tr_n=as_.fit_transform(y_tr.reshape(-1,1)).ravel()
y_val_n=as_.transform(y_val.reshape(-1,1)).ravel(); y_te_n=as_.transform(y_te.reshape(-1,1)).ravel()

# Store physical-unit bounds as tensors for denormalization inside training
bounds_low = torch.tensor(PARAM_BOUNDS[:, 0], dtype=torch.float32).to(device)
bounds_high = torch.tensor(PARAM_BOUNDS[:, 1], dtype=torch.float32).to(device)
abs_min = torch.tensor(as_.data_min_[0], dtype=torch.float32).to(device)
abs_range = torch.tensor(as_.data_range_[0], dtype=torch.float32).to(device)

BS = 256
def mkl(a,b,bs,sh=True): return DataLoader(TensorDataset(torch.tensor(a,dtype=torch.float32),torch.tensor(b,dtype=torch.float32)),batch_size=bs,shuffle=sh,pin_memory=True,num_workers=0)
fwd_tr=mkl(X_tr_n,y_tr_n,BS); fwd_vl=mkl(X_val_n,y_val_n,BS,False)
inv_tr=mkl(y_tr_n.reshape(-1,1),X_tr_n,BS); inv_vl=mkl(y_val_n.reshape(-1,1),X_val_n,BS,False)
print(f'Train:{X_tr.shape[0]} Val:{X_val.shape[0]} Test:{X_te.shape[0]}')

## Forward Surrogate (Phase 1)

In [ ]:
class ForwardSurrogate(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(20,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
    def forward(self,x): return self.net(x).squeeze(-1)

surrogate=ForwardSurrogate().to(device)
opt_s=optim.Adam(surrogate.parameters(),lr=1e-3); crit=nn.MSELoss()
print('Training surrogate ...')
for ep in range(1,101):
    surrogate.train()
    for xb,yb in fwd_tr:
        xb,yb=xb.to(device),yb.to(device); l=crit(surrogate(xb),yb); opt_s.zero_grad(); l.backward(); opt_s.step()
for p in surrogate.parameters(): p.requires_grad=False
surrogate.eval()
with torch.no_grad(): vp=surrogate(torch.tensor(X_val_n,dtype=torch.float32).to(device)).cpu().numpy()
print(f'Surrogate val R²={r2_score(y_val_n,vp):.4f}')

## PINN Inverse Model

In [ ]:
class PINNInverse(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 256),  nn.LeakyReLU(0.2), nn.BatchNorm1d(256),
            nn.Linear(256, 256), nn.LeakyReLU(0.2), nn.BatchNorm1d(256), nn.Dropout(0.15),
            nn.Linear(256, 128), nn.LeakyReLU(0.2), nn.BatchNorm1d(128),
            nn.Linear(128, 20), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


model = PINNInverse().to(device)
print(f'PINN parameters: {sum(p.numel() for p in model.parameters()):,}')

## Training with Composite Loss

In [ ]:
EPOCHS = 250
W1_PARAM = 0.5     # param MSE weight
W2_SURR  = 0.3     # surrogate recon weight
W3_MAX   = 0.2     # max physics weight
PHYS_WARMUP = 30   # epochs before physics loss reaches full weight

optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

train_losses, val_losses = [], []
param_l, surr_l, phys_l = [], [], []
best_val, best_state = float('inf'), None

print('Training PINN inverse model ...')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    w3 = W3_MAX * min(1.0, epoch / PHYS_WARMUP)

    model.train()
    r_tot, r_p, r_s, r_ph, n = 0., 0., 0., 0., 0
    for ab, par in inv_tr:
        ab, par = ab.to(device), par.to(device)
        pred_n = model(ab)  # (B, 20) normalised

        # 1) Parameter MSE
        loss_param = crit(pred_n, par)

        # 2) Surrogate reconstruction
        loss_surr = crit(surrogate(pred_n), ab.squeeze(-1))

        # 3) Physics loss: denormalise predicted params to physical units, run torch TMM
        pred_phys = pred_n * (bounds_high - bounds_low) + bounds_low  # (B, 20)
        target_abs_phys = ab.squeeze(-1) * abs_range + abs_min  # (B,) real absorption
        pred_abs_phys = tmm_torch(pred_phys)  # (B,)
        loss_phys = crit(pred_abs_phys, target_abs_phys)

        loss = W1_PARAM * loss_param + W2_SURR * loss_surr + w3 * loss_phys
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        bs = ab.size(0)
        r_tot += loss.item()*bs; r_p += loss_param.item()*bs
        r_s += loss_surr.item()*bs; r_ph += loss_phys.item()*bs; n += bs

    train_losses.append(r_tot/n)
    param_l.append(r_p/n); surr_l.append(r_s/n); phys_l.append(r_ph/n)

    model.eval()
    vr, vn = 0., 0
    with torch.no_grad():
        for ab, par in inv_vl:
            ab, par = ab.to(device), par.to(device)
            pp = model(ab)
            pp_phys = pp*(bounds_high-bounds_low)+bounds_low
            t_phys = ab.squeeze(-1)*abs_range+abs_min
            vl = W1_PARAM*crit(pp,par) + W2_SURR*crit(surrogate(pp),ab.squeeze(-1)) + w3*crit(tmm_torch(pp_phys),t_phys)
            vr += vl.item()*ab.size(0); vn += ab.size(0)
    val_losses.append(vr/vn)
    scheduler.step()

    if val_losses[-1] < best_val:
        best_val = val_losses[-1]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 50 == 0:
        print(f'  Epoch {epoch:3d}  total={train_losses[-1]:.6f}  '
              f'param={param_l[-1]:.6f}  surr={surr_l[-1]:.6f}  phys={phys_l[-1]:.6f}  '
              f'val={val_losses[-1]:.6f}  w3={w3:.3f}')

train_time = time.time() - t0
model.load_state_dict(best_state); model.eval()
print(f'Done in {train_time:.1f}s. Best val={best_val:.6f}')

## Evaluation

In [ ]:
with torch.no_grad():
    pred_n = model(torch.tensor(y_te_n.reshape(-1,1),dtype=torch.float32).to(device)).cpu().numpy()

param_mse = mean_squared_error(X_te_n, pred_n)
param_mae = mean_absolute_error(X_te_n, pred_n)
param_r2  = r2_score(X_te_n, pred_n)
print(f'Param-space  MSE={param_mse:.6f}  MAE={param_mae:.6f}  R²={param_r2:.4f}')

In [ ]:
N_VAL = 200
pred_phys = ps.inverse_transform(pred_n[:N_VAL])
target_abs = y_te[:N_VAL]
recon = np.array([tmm_forward(p) for p in pred_phys])

recon_mse = mean_squared_error(target_abs, recon)
recon_mae = mean_absolute_error(target_abs, recon)
recon_r2  = r2_score(target_abs, recon)
print(f'TMM Recon  MSE={recon_mse:.6f}  MAE={recon_mae:.4f}  R²={recon_r2:.4f}')

## Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1) Training curves
ax=axes[0,0]; ax.plot(train_losses,label='Train'); ax.plot(val_losses,label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Composite Loss'); ax.set_title('Total Loss'); ax.legend()

# 2) Loss components
ax=axes[0,1]
ax.plot(param_l,label='Param MSE'); ax.plot(surr_l,label='Surrogate Recon'); ax.plot(phys_l,label='Physics')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Loss Components'); ax.legend()

# 3) Reconstruction scatter
ax=axes[1,0]
ax.scatter(target_abs,recon,s=8,alpha=.6); ax.plot([0,1],[0,1],'r--')
ax.set_xlabel('Target'); ax.set_ylabel('TMM Recon'); ax.set_title(f'Physics Validation (R²={recon_r2:.3f})')

# 4) Per-param error
ax=axes[1,1]
ppm=np.abs(X_te_n[:N_VAL]-pred_n[:N_VAL]).mean(axis=0)
ax.bar(range(20),ppm); ax.set_xticks(range(20))
ax.set_xticklabels(PARAM_COLUMNS,rotation=90,fontsize=7)
ax.set_ylabel('MAE (norm)'); ax.set_title('Per-Parameter Error')

plt.tight_layout(); plt.show()

In [ ]:
# Error histogram
errors = np.abs(target_abs - recon)
plt.figure(figsize=(6,4))
plt.hist(errors, bins=30, edgecolor='k')
plt.xlabel('|Target - Recon| Absorption'); plt.ylabel('Count')
plt.title(f'Recon Error (mean={errors.mean():.4f})'); plt.tight_layout(); plt.show()

## Results Summary

In [ ]:
results = {
    'method': 'PINN',
    'param_mse': float(param_mse),
    'param_mae': float(param_mae),
    'param_r2': float(param_r2),
    'recon_mse': float(recon_mse),
    'recon_mae': float(recon_mae),
    'recon_r2': float(recon_r2),
    'can_generate_diverse': False,
    'training_time_seconds': round(train_time, 1),
}
print(results)

### Discussion

The PINN approach directly embeds the Transfer-Matrix Method physics into the training loss. This provides a hard constraint that predicted designs must satisfy the acoustic wave equations, not just match the data distribution.

**Advantages**: Physics-guaranteed consistency; works well when data is limited; the differentiable TMM can be re-used for gradient-based design optimisation.

**Limitations**: The differentiable TMM is computationally expensive (even at 10 frequencies); training is slower per epoch; deterministic output (one design per target); capturing full 1000-frequency physics is impractical during training.